In [1]:
from langchain_groq import ChatGroq
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import yfinance as yf
from langchain.agents import AgentType, initialize_agent
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool
from langchain_groq import ChatGroq

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from pymongo.operations import SearchIndexModel
from mongodb_atlas import MongoDBAtlas, MongoDBAtlasDocumentManager
from langchain_community.vectorstores import AtlasDB
# from langchain_mongodb import MongoDBAtlasVectorSearch
# from pymongo import MongoClient
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser
import PyPDF2
import pandas as pd
from langchain_core.documents import Document
from typing import Iterable
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter
)
from langchain.text_splitter import NLTKTextSplitter
from mongodb_atlas import MongoDBAtlas, MongoDBAtlasDocumentManager
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from mongodb_atlas import MongoDBAtlas, MongoDBAtlasDocumentManager
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from file_reader import FileReader
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables.utils import ConfigurableFieldSpec
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

USER_AGENT environment variable not set, consider setting it to identify your requests.
/home/steeldev/Desktop/Github/PotatoTrade/mlenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

# Tools to be used

In [5]:
import requests

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = 'https://www.alphavantage.co/query?function=OVERVIEW&symbol=IBM&apikey='+os.getenv('ALPHA_VANTAGE_API_KEY')
r = requests.get(url)
data = r.json()

# url2='https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=IBM&interval=1min&outputsize=full&apikey='+os.getenv('ALPHA_VANTAGE_API_KEY')
# r2 = requests.get(url2)
# data2 = r2.json()

print(data)

{'Symbol': 'IBM', 'AssetType': 'Common Stock', 'Name': 'International Business Machines', 'Description': 'International Business Machines Corporation (IBM) is an American multinational technology company headquartered in Armonk, New York, with operations in over 170 countries. The company began in 1911, founded in Endicott, New York, as the Computing-Tabulating-Recording Company (CTR) and was renamed International Business Machines in 1924. IBM is incorporated in New York. IBM produces and sells computer hardware, middleware and software, and provides hosting and consulting services in areas ranging from mainframe computers to nanotechnology. IBM is also a major research organization, holding the record for most annual U.S. patents generated by a business (as of 2020) for 28 consecutive years. Inventions by IBM include the automated teller machine (ATM), the floppy disk, the hard disk drive, the magnetic stripe card, the relational database, the SQL programming language, the UPC barcod

In [6]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("what is hunter x hunter")

'Hunter x Hunter tells the tale of Gon who aims to become a hunter and go on an adventure to find his father. Over time, Gon\'s expedition will unfold into a much grander ride with lots of engrossing characters and fascinating storylines. The HxH manga began back in 1998 and is still ongoing to this day. Key visual for the series. Hunter × Hunter is an anime television series that aired from 2011 to 2014 based on Yoshihiro Togashi\'s manga series Hunter × Hunter.The story begins with a young boy named Gon Freecss, who one day discovers that the father who he thought was dead, is in fact alive and well.He learns that his father, Ging, is a legendary "Hunter", an individual who has proven ... Hunter x Hunter takes place in a fantastical world filled with dreadful monsters, menacing beasts, and all manner of hidden treasures.Whereas other series with similar settings present something of a free-for-all of bandits and brigands, Hunter x Hunter presents a slightly more regulated landscape.I

In [7]:
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wikipedia.run("HUNTER X HUNTER")

'Page: Hunter × Hunter\nSummary: Hunter × Hunter (pronounced "hunter hunter") is a Japanese manga series written and illustrated by Yoshihiro Togashi. It has been serialized in Shueisha\'s shōnen manga magazine Weekly Shōnen Jump since March 1998, although the manga has frequently gone on extended hiatuses since 2006. Its chapters have been collected in 38 tankōbon volumes as of September 2024. The story focuses on a young boy named Gon Freecss who discovers that his father, who left him at a young age, is actually a world-renowned Hunter, a licensed professional who specializes in fantastical pursuits such as locating rare or unidentified animal species, treasure hunting, surveying unexplored enclaves, or hunting down lawless individuals. Gon departs on a journey to become a Hunter and eventually find his father. Along the way, Gon meets various other Hunters and encounters the paranormal.\nHunter × Hunter was adapted into a 62-episode anime television series by Nippon Animation and d

# Classes

In [ ]:
DB_NAME = "potato_trade"
COLLECTION_NAME = "vectorized_documents"

atlas = MongoDBAtlas(DB_NAME, COLLECTION_NAME)
document_manager = MongoDBAtlasDocumentManager(atlas=atlas)

In [ ]:


class FileReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.content = None
    
    def read(self):
        if self.file_path.endswith(".pdf"):
            self.content = self._read_pdf()
        elif self.file_path.endswith(".txt"):
            self.content = self._read_txt()
        elif self.file_path.endswith(".csv"):
            self.content = self._read_csv()
        else:
            raise ValueError("Unsupported file format")
        self.content= [Document(
            page_content=self.content,
            metadata={"source": self.file_path}
        )]
        return self.content
    
    def _read_pdf(self):
        text = ""
        with open(self.file_path, "rb") as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text += page.extract_text() + "\n"
        return text.strip()
    
    def _read_txt(self):
        with open(self.file_path, "r", encoding="utf-8") as file:
            return file.read().strip()
    
    def _read_csv(self):
        df = pd.read_csv(self.file_path)
        return df.to_string(index=False)


In [4]:
fl=FileReader("../Data/stock_docs/FIL_Stock Market.pdf")
fl.read()

[Document(metadata={'source': '../Data/stock_docs/FIL_Stock Market.pdf'}, page_content='Basics Of Stock Market \nBy  Ronak Nangalia \nSohrab Kothari \nInvestment & Need of Investment \n•The money you earn is partly spent and the rest sav ed for \nmeeting future expenses. Instead of keeping the sav ings idle \nyou may like to use savings in order to get return on it in the \nfuture. This is called Investment. \n•One needs to invest to \n1. earn return on your idle resources 2. generate a specified sum of money for a specific goal in \nlife \n3. make a provision for an uncertain future \nWhen to Start Investing \n•The sooner one starts investing the better. By investing early you allow your investments more tim e \nto grow, increases your income, by accumulating the  \nprincipal and the interest or dividend earned on it , \nyear after year. \nyear after year. \n•The three golden rules for all investors are: \n1. Invest early 2. Invest regularly 3. Invest for long term and not short term 

In [5]:


class TextProcessor:
    def __init__(self, text, chunk_size=500, chunk_overlap=50, split_method="recursive"):
        self.text = text
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.split_method = split_method
        self.chunks = None
        self.embeddings = None
        self.embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.embeddings_dict = {}  # Store embeddings as key-value pairs
    
    def split_text(self):
        try:
            if self.split_method == "recursive":
                splitter = RecursiveCharacterTextSplitter(
                    chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap
                )
            elif self.split_method == "character":
                splitter = CharacterTextSplitter(
                    chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap
                )
            elif self.split_method == "token":
                splitter = TokenTextSplitter(chunk_size=self.chunk_size)
            elif self.split_method == "nltk":
                splitter = NLTKTextSplitter()
            else:
                raise ValueError("Invalid split method. Choose from 'recursive', 'character', 'token', or 'nltk'.")
            
            self.chunks = splitter.split_text(self.text)
            return self.chunks
        except Exception as e:
            print(f"Error in text splitting: {e}")
            return None
    
    def generate_embeddings(self, batch_size=32, normalize=False):
        try:
            if self.chunks is None:
                raise ValueError("Text must be split before generating embeddings.")
            
            embeddings = self.embedding_model.encode(
                self.chunks, convert_to_tensor=True, batch_size=batch_size, normalize_embeddings=normalize
            )
            
            self.embeddings_dict = {chunk: embedding for chunk, embedding in zip(self.chunks, embeddings)}
            return self.embeddings_dict
        except Exception as e:
            print(f"Error in generating embeddings: {e}")
            return None


/home/steeldev/Desktop/Github/PotatoTrade/mlenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
# TP = TextProcessor(fl.content, chunk_size=300, chunk_overlap=50, split_method="recursive")
# TP.split_text()
# TP.generate_embeddings()

In [ ]:


class MongoUploader:

    def __init__(self, db_name: str, collection_name: str,document: List[Document], model_name: str="sentence-transformers/all-MiniLM-L6-v2",
                  vector_search_index: str="test_vector_index", split_method: str="recursive", chunk_size: int=500, chunk_overlap: int=50
                 ):
        self.db_name = db_name
        self.collection_name = collection_name
        self.atlas = MongoDBAtlas(db_name, collection_name)
        self.vector_search_index = vector_search_index
        self.document_manager = MongoDBAtlasDocumentManager(atlas=self.atlas)
        self.model_name = model_name
        self.embedding = HuggingFaceEmbeddings(model_name=model_name)
        self.document= document
        self.checkpoint_split_document=None
        self.split_method = split_method
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.docs_split_by_checkpoint = None
        self.docs_split_by_splitter=None

    def execute(self):
        self.atlas.create_vector_store(
            embedding=self.embedding,
            index_name=self.vector_search_index,
            relevance_score_fn="cosine"
        )
        self.docs_split_by_checkpoint= self.document_manager.split_documents(
            documents=self.document,
            split_condition=self._split_by_checkpoint, split_index_name="doc_index"
        )
        self.splitter= self._get_splitter()
        self.docs_split_by_splitter= self.document_manager.split_documents_by_splitter(
            self.splitter, self.docs_split_by_checkpoint
        )
        for index, doc in enumerate(self.docs_split_by_splitter):
            self.doc.metadata.update({"chunk_index": index})

        ids=self.atlas.add_documents(
            documents=self.docs_split_by_splitter
        )

    def _split_by_checkpoint(self) -> List[str]:
        checkpoints= self.document.split("[ CHECKPOINT")
        self.checkpoint_split_document= [checkpoints.split(" ]",1)[-1].strip() for checkpoints in checkpoints]
        return self.checkpoint_split_document
    
    def _get_splitter(self):
        try:
            if self.split_method == "recursive":
                splitter = RecursiveCharacterTextSplitter(
                    chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap
                )
            elif self.split_method == "character":
                splitter = CharacterTextSplitter(
                    chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap
                )
            elif self.split_method == "token":
                splitter = TokenTextSplitter(chunk_size=self.chunk_size)
            elif self.split_method == "nltk":
                splitter = NLTKTextSplitter()
            else:
                raise ValueError("Invalid split method. Choose from 'recursive', 'character', 'token', or 'nltk'.")
            
            
            return splitter
        except Exception as e:
            print(f"Error in text splitting: {e}")
            return None

# Non class code

In [ ]:

DB_NAME = "potato_trade"
COLLECTION_NAME = "vectorized_documents"

atlas = MongoDBAtlas(DB_NAME, COLLECTION_NAME)
document_manager = MongoDBAtlasDocumentManager(atlas=atlas)
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

TUTORIAL_VECTOR_SEARCH_INDEX_NAME = "test_vector_index"

atlas.create_vector_store(
    embedding=embedding,
    index_name=TUTORIAL_VECTOR_SEARCH_INDEX_NAME,
    relevance_score_fn="cosine",
)


def split_by_chapter(text: str) -> List[str]:
    chapters = text.split("[ CHECKPOINT")
    return [chapter.split(" ]", 1)[-1].strip() for chapter in chapters]

split_chapters = document_manager.split_documents(
    fl.read(), split_condition=split_by_chapter, split_index_name="doc_index"
)
first_chapter = split_chapters[0]
print(f"{first_chapter.page_content[:80]}, metadata: {first_chapter.metadata}")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
split_documents = document_manager.split_documents_by_splitter(
    text_splitter, split_chapters
)
for index, doc in enumerate(split_documents):
    doc.metadata.update({"chunk_index": index})

In [57]:
ids = atlas.add_documents(documents=split_documents)

In [58]:
ids

['67e568f11644c9727980c5ca',
 '67e568f11644c9727980c5cb',
 '67e568f11644c9727980c5cc',
 '67e568f11644c9727980c5cd',
 '67e568f11644c9727980c5ce',
 '67e568f11644c9727980c5cf',
 '67e568f11644c9727980c5d0',
 '67e568f11644c9727980c5d1',
 '67e568f11644c9727980c5d2',
 '67e568f11644c9727980c5d3',
 '67e568f11644c9727980c5d4']

In [59]:
atlas.similarity_search(query="stock market", k=2)

[Document(metadata={'_id': '67e4cdb00e719b3b7e644dfe', 'doc_index': 0, 'chunk_index': 2}, page_content='The share market, also known as the stock market, is a platform where buyers and sellers come together to trade publicly listed shares of companies. The market is regulated by the Securities and Exchange Board of India (SEBI), which oversees the functioning of stock exchanges and ensures that listed companies comply with regulations and disclosure requirements.\n\nIf a company has issued 100 shares and you own 1 share, you own 1% stake in the company. The share market is where shares of different companies are traded.'),
 Document(metadata={'_id': '67e4cdb00e719b3b7e644dfc', 'doc_index': 0, 'chunk_index': 0}, page_content='Investing is a way to grow your money over time by putting it into various assets. One of the popular investment options is stocks. When you invest in stocks, you become a shareholder and can benefit from the company’s profits and growth. However, the stock market 

In [ ]:

class MongoUploader:

    def __init__(self, db_name: str, collection_name: str,document_path: str, model_name: str="sentence-transformers/all-MiniLM-L6-v2",
                  vector_search_index: str="test_vector_index", split_method: str="recursive", chunk_size: int=500, chunk_overlap: int=50
                 ):
        self.db_name = db_name
        self.collection_name = collection_name
        self.atlas = MongoDBAtlas(db_name, collection_name)
        self.vector_search_index = vector_search_index
        self.document_manager = MongoDBAtlasDocumentManager(atlas=self.atlas)
        self.model_name = model_name
        self.document= None
        self.document_path= document_path
        self.checkpoint_split_document=None
        self.split_method = split_method
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.docs_split_by_checkpoint = None
        self.docs_split_by_splitter=None
        self.embedding = HuggingFaceEmbeddings(model_name=model_name)
        self._file_reader= FileReader(self.document_path)

    def execute(self):
        self.document=self._file_reader.read()
        # return self.document
        self.atlas.create_vector_store(
            embedding=self.embedding,
            index_name=self.vector_search_index,
            relevance_score_fn="cosine"
        )
        self.docs_split_by_checkpoint= self.document_manager.split_documents(
            documents=self.document,
            split_condition=self._split_by_checkpoint, split_index_name="doc_index"
        )
        # print(self.docs_split_by_checkpoint)
        self.splitter= self._get_splitter()
        self.docs_split_by_splitter= self.document_manager.split_documents_by_splitter(
            self.splitter, self.docs_split_by_checkpoint
        )
        for index, doc in enumerate(self.docs_split_by_splitter):
            doc.metadata.update({"chunk_index": index})
        print(self.docs_split_by_splitter)
        ids=self.atlas.add_documents(
            documents=self.docs_split_by_splitter
        )
        return ids

    def _split_by_checkpoint(self, text: str) -> List[str]:
        chapters = text.split("[ CHECKPOINT")
        return [chapter.split(" ]", 1)[-1].strip() for chapter in chapters]
        # return self.checkpoint_split_document
    
    def _get_splitter(self):
        try:
            if self.split_method == "recursive":
                splitter = RecursiveCharacterTextSplitter(
                    chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap
                )
            elif self.split_method == "character":
                splitter = CharacterTextSplitter(
                    chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap
                )
            elif self.split_method == "token":
                splitter = TokenTextSplitter(chunk_size=self.chunk_size)
            elif self.split_method == "nltk":
                splitter = NLTKTextSplitter()
            else:
                raise ValueError("Invalid split method. Choose from 'recursive', 'character', 'token', or 'nltk'.")
            
            
            return splitter
        except Exception as e:
            print(f"Error in text splitting: {e}")
            return None

In [81]:
mgu= MongoUploader("potato_trade", "vectorized_documents", "../Data/stock_docs/gpt.txt")
mgu.execute()

[Document(metadata={'doc_index': 0, 'chunk_index': 0}, page_content='A stock represents ownership in a company. When you own stock in a company, you become a shareholder, meaning you have a claim on a portion of the company’s assets and earnings.\n\nStocks are also referred to as equities because they represent a form of equity ownership.\n\nCompanies issue stocks to raise capital for expansion, research, and other business activities.\n\nA share is a single unit of stock.'), Document(metadata={'doc_index': 0, 'chunk_index': 1}, page_content='A share is a single unit of stock.\n\nWhen a company issues stock, it is divided into a certain number of shares, and each share represents a fractional ownership in the company.\n\nFor example, if a company has issued 1 million shares and you own 1,000 shares, you own 0.1% of the company.\n\nEquity refers to the ownership interest in a company, represented by stocks.\n\nIt is the difference between a company’s total assets and total liabilities.'

['67e57e881644c9727980c5df',
 '67e57e881644c9727980c5e0',
 '67e57e881644c9727980c5e1',
 '67e57e881644c9727980c5e2',
 '67e57e881644c9727980c5e3',
 '67e57e881644c9727980c5e4',
 '67e57e881644c9727980c5e5',
 '67e57e881644c9727980c5e6',
 '67e57e881644c9727980c5e7',
 '67e57e881644c9727980c5e8',
 '67e57e881644c9727980c5e9',
 '67e57e881644c9727980c5ea',
 '67e57e881644c9727980c5eb',
 '67e57e881644c9727980c5ec']

In [ ]:

class MongoDBAtlasQA:
    def __init__(self, mongo_uri, db_name, collection_name, embedding, index_name, llm, session_id: str = "test_session"):
        """Initialize the MongoDB Atlas QA system."""
        try:
            self.atlas= MongoDBAtlas(db_name, collection_name)
            self.document_manager= MongoDBAtlasDocumentManager(atlas=self.atlas)  
            self.atlas.create_vector_store(
                embedding=embedding,
                index_name=index_name,
                relevance_score_fn="cosine",
            )
            # llm: ChatGroq=llm
            self.llm=llm
            self.llm_with_tools=self.llm.bind_tools([self.get_similarity_search])
            self.llm_with_tools_and_output_parser= self.llm_with_tools | StrOutputParser()
            # self.session_id= session_id
            # self._chat_message_history=SQLChatMessageHistory(
            #     session_id=self.session_id,
            #     connection=os.getenv("POSTGRES_URI"),
            # )
            # self._config_fields = [
            #     ConfigurableFieldSpec(
            #         id="user_id",
            #         annotation=str,
            #         name="User ID",
            #         description="Unique identifier for a user.",
            #         default="",
            #         is_shared=True,
            #     ),
            #     ConfigurableFieldSpec(
            #         id="conversation_id",
            #         annotation=str,
            #         name="Conversation ID",
            #         description="Unique identifier for a conversation.",
            #         default="",
            #         is_shared=True,
            #     ),
            # ]
            # llm_with_history=RunnableWithMessageHistory(
            #     llm_with_tools_and_output_parser,
            #     message_history=self.get_chat_history,
            #     input_messages_key="question",
            #     history_messages_key="chat_history",
            #     # Set parameters for retrieving chat history
            #     history_factory_config=self._config_fields
            # )
            self.llm= self.llm_with_tools_and_output_parser
        except Exception as e:
            print(f"Error initializing MongoDBAtlasQA: {e}")

    @tool
    def get_similarity_search(self, query):
        """Perform a similarity search for info on stock market."""
        try:
            results = self.atlas.similarity_search(query)
            return results
        except Exception as e:
            print(f"Error performing similarity search: {e}")
            return None

    def get_chat_history(self, user_id, conversation_id):
        try:
            return self._chat_message_history
        except Exception as e:  
            print(f"Error retrieving chat history: {e}")
            return None
    
    def query(self, text):
        """Query the knowledge base."""
        try:
            # human_message = HumanMessage(content=text)
            # self._chat_message_history.add_message(human_message)  
            result=self.llm.invoke(text)
            # ai_message = AIMessage(content=result)
            # self._chat_message_history.add_message(ai_message)
            return result
        except Exception as e:
            print(f"Error querying the knowledge base: {e}")
            return None


In [226]:

class MongoDBAtlasQA:
    def __init__(self, mongo_uri, db_name, collection_name, embedding, index_name, llm, conversation_id: str= "test_session"):
        """Initialize the MongoDB Atlas QA system."""
        try:
            self.atlas= MongoDBAtlas(db_name, collection_name)
            self.document_manager= MongoDBAtlasDocumentManager(atlas=self.atlas)  
            self.atlas.create_vector_store(
                embedding=embedding,
                index_name=index_name,
                relevance_score_fn="cosine",
            )
            self._prompt = ChatPromptTemplate.from_messages(
                [
                    ("system", "You are a helpful assistant."),
                    # Placeholder for chat history
                    MessagesPlaceholder(variable_name="chat_history"),
                    ("human", "{query}"),
                ]
            )
            self.llm: ChatGroq=llm
            self.llm.bind_tools([self.get_similarity_search])
            self.llm= self._prompt| self.llm | StrOutputParser()
            
            self._config_fields = [
                ConfigurableFieldSpec(
                    id="user_id",
                    annotation=str,
                    name="User ID",
                    description="Unique identifier for a user.",
                    default="",
                    is_shared=True,
                ),
                ConfigurableFieldSpec(
                    id="conversation_id",
                    annotation=str,
                    name="Conversation ID",
                    description="Unique identifier for a conversation.",
                    default="",
                    is_shared=True,
                ),
            ]
            self.llm_with_history=RunnableWithMessageHistory(
                self.llm,
                self.get_chat_history,
                input_messages_key="query",
                history_messages_key="chat_history",
                history_factory_config=self._config_fields
            )
            self.config = {"configurable": {"user_id": "user1", "conversation_id": conversation_id}}
        except Exception as e:
            print(f"Error initializing MongoDBAtlasQA: {e}")

    def get_chat_history(self, user_id, conversation_id):
        try:
            return SQLChatMessageHistory(
                table_name="basic_stock_info_chat_history",
                session_id=conversation_id,
                connection=os.getenv("POSTGRES_URI"),
            )
            
        except Exception as e:  
            print(f"Error retrieving chat history: {e}")
            return None

    @tool
    def get_similarity_search(self, query):
        """Perform a similarity search for info on stock market."""
        try:
            results = self.atlas.similarity_search(query)
            return results
        except Exception as e:
            print(f"Error performing similarity search: {e}")
            return None

    def query(self, text):
        """Query the knowledge base."""
        try:
            return self.llm_with_history.invoke(
                {"query": text}, self.config
            )
        except Exception as e:
            print(f"Error querying the knowledge base: {e}")
            return None
    
    

In [ ]:
MONGO_URI = os.getenv("MONGODB_ATLAS_CLUSTER_URI")
DB_NAME = "potato_trade"
COLLECTION_NAME = "vectorized_documents"
INDEX_NAME = "test_vector_index"

llm = ChatGroq(model_name="llama3-8b-8192", api_key=os.getenv("GROQ_API_KEY"), async_client=True)

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
qa_system = MongoDBAtlasQA(MONGO_URI, DB_NAME, COLLECTION_NAME, embedding, INDEX_NAME, llm)

query_text = "What is the stock market? explain in points"

response =  qa_system.query(query_text)
print(response)

/home/steeldev/Desktop/Github/PotatoTrade/mlenv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Here's an explanation of the stock market in points:

**What is the Stock Market?**

* The stock market, also known as the equity market or share market, is a platform where publicly traded companies' shares are bought and sold.
* It's a place where investors can buy and sell parts of companies, giving them ownership and potential profit.

**Key Points:**

1. **Publicly Traded Companies**: Only companies that are listed on a stock exchange can have their shares traded on the market.
2. **Stock Exchanges**: There are several stock exchanges around the world, such as the New York Stock Exchange (NYSE), NASDAQ, and the London Stock Exchange (LSE).
3. **Listed Securities**: Shares of publicly traded companies are listed on an exchange, making them available for trading.
4. **Investors**: Individuals, institutions, and organizations can buy and sell shares of listed companies through a brokerage account.
5. **Stock Prices**: The price of a share is determined by supply and demand in the mar

In [238]:
response =  qa_system.query("Explain it in short")
print(response)

Dollar-Cost Averaging (DCA) is a investing strategy where you invest a fixed amount of money at regular intervals, regardless of the market's performance. This helps reduce timing risk, encourages discipline, and smooths out market volatility.


In [ ]:


# Initialize chat history with session ID and database connection.
chat_message_history = SQLChatMessageHistory(
    session_id="test_session", connection=os.getenv("POSTGRES_URI")
)

In [55]:
# Add a user message
chat_message_history.add_user_message(
    "Hello, nice to meet you! My name is Heesun :) I'm a LangChain developer. I look forward to working with you!"
)
# Add an AI message
chat_message_history.add_ai_message(
    "Hi, Heesun! Nice to meet you. I look forward to working with you too!"
)

In [56]:
# Clear the session memory
chat_message_history.clear()
chat_message_history.messages

[]